# Week 1 · Day 2 — Lab 5
## Reproducible Randomness + End-to-End Capstone

Randomness is everywhere in AI work — weight init, dropout, shuffling, sampling,
synthetic data, train/test splits. If it isn't **reproducible**, your bugs aren't
either. NumPy 2.x's answer is the **`Generator`** API created by
`np.random.default_rng(seed)`. The legacy `np.random.seed()` / `np.random.rand()`
global functions still exist but are discouraged: they share one hidden global
state, which is exactly what makes "random" results impossible to reproduce.

This lab has two halves. First (~25 min) you drill the modern RNG: seeding,
distributions, sampling, shuffling, independent streams via `spawn`. Then
(~40 min) a **capstone** ties the whole day together — load-free synthetic eval
data, NaN-safe stats, normalization, masking, ranking, a cosine-similarity
retrieval step, and a saved results bundle.

### Learning objectives
1. Create seeded generators with `np.random.default_rng` and explain why global `np.random.*` is discouraged.
2. Draw from the common distributions (`random`, `standard_normal`, `uniform`, `integers`).
3. Sample without replacement (`choice`) and shuffle reproducibly.
4. Produce independent, non-overlapping streams with `Generator.spawn`.
5. **Capstone:** combine reductions, NaN-safe stats, broadcasting, masking, ranking, a little linear algebra, and `savez_compressed` into one analysis.

### Time budget — ~70 min
| Segment | Time |
|---|---|
| Framing & objectives | 4 min |
| **A.** Seeding & reproducibility | 8 min |
| **B.** Distributions | 7 min |
| **C.** Sampling & shuffling | 6 min |
| **D.** Independent streams (`spawn`) | 6 min |
| **CAP.** End-to-end eval pipeline | 36 min |
| Wrap-up | 3 min |

### Files you need
**None** — this lab generates everything from seeds. It *writes* one output
bundle (`eval_results.npz`) at the end.


In [ ]:
import numpy as np
from pathlib import Path

print("NumPy", np.__version__)   # target curriculum: NumPy 2.x on Python 3.13

OUT = Path("data")
if not OUT.exists():
    OUT = Path(".")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

# This lab GENERATES its own data with the modern Generator API — that is the
# lesson. There are no provided .npy inputs to load.
print("ready — this lab makes its own random data, reproducibly.")

## Part A — Seeding & reproducibility  *(guided)*

`np.random.default_rng(seed)` returns a `Generator`. Two generators built from
the **same seed** produce the **same stream** — that's reproducibility. The old
`np.random.seed(...)` + `np.random.rand(...)` pattern mutates one global state
shared across your whole process, so any other code that draws a random number
shifts your results. Prefer an explicit generator object you pass around.


In [ ]:
rng = np.random.default_rng(42)
print("three draws:", rng.random(3).round(4))

a = np.random.default_rng(7).random(3)
b = np.random.default_rng(7).random(3)
print("same seed -> identical stream:", np.array_equal(a, b))

### Exercise A1 — Prove reproducibility
Make two generators from seed `2025`, draw 5 floats from each into `draw_a` and
`draw_b`, and confirm they're identical. Then make a generator from a *different*
seed (`2026`) into `draw_c` and confirm it differs.


💡 **Hint.** `np.random.default_rng(2025).random(5)` — call it twice with the
same seed for a/b, once with `2026` for c.


In [ ]:
draw_a = None   # TODO: 5 floats from default_rng(2025)
draw_b = None   # TODO: 5 floats from a SECOND default_rng(2025)
draw_c = None   # TODO: 5 floats from default_rng(2026)

In [ ]:
check("A1: same seed -> identical streams", lambda: np.array_equal(draw_a, draw_b))
check("A1: different seed -> different stream", lambda: not np.array_equal(draw_a, draw_c))

## Part B — Distributions

The `Generator` exposes the draws you'll use constantly:
- `rng.random(size)` — uniform in [0, 1),
- `rng.standard_normal(size)` — standard Gaussian (mean 0, std 1),
- `rng.uniform(low, high, size)` — uniform in [low, high),
- `rng.integers(low, high, size)` — random ints (high exclusive by default).


In [ ]:
rng = np.random.default_rng(0)
print("uniform[0,1):", rng.random(3).round(3))
print("gaussian    :", rng.standard_normal(3).round(3))
print("uniform 5-10:", rng.uniform(5, 10, 3).round(3))
print("ints 0-99   :", rng.integers(0, 100, 5))

### Exercise B1 — A synthetic batch
From one generator seeded `123`, draw:
- `latencies` — 1000 values uniform in [50, 250) (milliseconds),
- `noise` — 1000 standard-normal values,
- `token_counts` — 1000 integers in [10, 512).

Then report the mean latency (should land near 150).


In [ ]:
rng = np.random.default_rng(123)
latencies = None       # TODO: uniform [50, 250), size 1000
noise = None           # TODO: standard normal, size 1000
token_counts = None    # TODO: integers in [10, 512), size 1000

In [ ]:
check("B1: latencies in [50, 250)",
      lambda: float(latencies.min()) >= 50 and float(latencies.max()) < 250)
check("B1: token_counts in [10, 512)",
      lambda: int(token_counts.min()) >= 10 and int(token_counts.max()) < 512)
check("B1: noise looks standard-normal",
      lambda: abs(float(noise.mean())) < 0.1 and abs(float(noise.std()) - 1) < 0.1)

## Part C — Sampling & shuffling

- `rng.choice(a, size, replace=False)` — sample **without** replacement (a random
  subset; no element repeats).
- `rng.permutation(n)` — a fresh shuffled `arange(n)` (returns a new array).
- `rng.shuffle(arr)` — shuffle an array **in place**.


In [ ]:
rng = np.random.default_rng(1)
pool = np.arange(10)
print("subset of 4 (no repeats):", rng.choice(pool, size=4, replace=False))
print("permutation of 0..9     :", rng.permutation(10))

### Exercise C1 — A reproducible train/test split
You have 200 example indices. Using a generator seeded `99`, draw a
**without-replacement** sample of 40 indices as `test_idx`, then make `train_idx`
the remaining 160 (use `np.setdiff1d`). Confirm they don't overlap and cover all 200.


💡 **Hint.** `rng.choice(200, size=40, replace=False)` gives the test indices;
`np.setdiff1d(np.arange(200), test_idx)` gives the rest.


In [ ]:
rng = np.random.default_rng(99)
all_idx = np.arange(200)
test_idx = None    # TODO: 40 unique indices, no replacement
train_idx = None   # TODO: the other 160 via np.setdiff1d

In [ ]:
check("C1: 40 test + 160 train",
      lambda: test_idx.shape[0] == 40 and train_idx.shape[0] == 160)
check("C1: no overlap between splits",
      lambda: np.intersect1d(test_idx, train_idx).size == 0)
check("C1: splits cover all 200 indices",
      lambda: np.array_equal(np.union1d(test_idx, train_idx), np.arange(200)))

## Part D — Independent streams with `spawn`

Need several generators that are guaranteed **not** to overlap (parallel workers,
per-fold splits, multiple augmentation pipelines)? Don't seed them with
`seed`, `seed+1`, `seed+2` — those streams can correlate. Use **`spawn`**, which
derives statistically independent child generators from a parent's seed sequence.


In [ ]:
parent = np.random.default_rng(2024)
kids = parent.spawn(3)                       # 3 independent child Generators
for i, k in enumerate(kids):
    print(f"child {i}:", k.random(3).round(4))

### Exercise D1 — Four independent workers, reproducibly
From a parent seeded `7`, spawn **4** children. Collect each child's 5 draws into
a list `streams` (a list of 4 arrays). Then prove reproducibility: respawn from a
*fresh* parent seeded `7`, take child 0's 5 draws as `replay`, and confirm it
equals `streams[0]`.


In [ ]:
parent = np.random.default_rng(7)
children = None     # TODO: parent.spawn(4)
streams = None      # TODO: [child.random(5) for child in children]

# Reproduce child 0 from a brand-new parent with the same seed:
replay = None       # TODO: np.random.default_rng(7).spawn(4)[0].random(5)

In [ ]:
check("D1: four independent streams", lambda: len(streams) == 4)
check("D1: each stream has 5 draws",
      lambda: all(s.shape == (5,) for s in streams))
check("D1: spawn is reproducible from the same parent seed",
      lambda: np.array_equal(streams[0], replay))
check("D1: distinct children differ",
      lambda: not np.array_equal(streams[0], streams[1]))

## CAPSTONE — End-to-end eval-score analysis  *(~36 min)*

This pulls the **entire day** together into one realistic task. You are handed a
batch of model responses, each with a few quality metrics (some missing) and an
embedding vector. Your job: clean, normalize, rank, retrieve, and persist.

You'll generate the data yourself (reproducibly), so the whole pipeline is
runnable anywhere with no external files. Work through the steps in order — each
builds on the previous one.


### CAP-0 — Generate the synthetic batch  *(guided — run as given)*
200 responses × 3 metrics, plus a 16-dim embedding per response. We inject ~5%
missing values into the metrics to exercise the NaN-safe path.


In [ ]:
rng = np.random.default_rng(42)
N, M, D = 200, 3, 16
METRICS = np.array(["relevance", "coherence", "safety"])

# Quality metrics in [0, 1], then knock out ~5% of cells as NaN.
eval_metrics = rng.uniform(0.3, 1.0, size=(N, M))
mask_missing = rng.random((N, M)) < 0.05
eval_metrics[mask_missing] = np.nan

# A 16-dim embedding per response (think: sentence embedding).
embeddings = rng.standard_normal((N, D))

print("eval_metrics:", eval_metrics.shape, "| missing cells:", int(np.isnan(eval_metrics).sum()))
print("embeddings  :", embeddings.shape)

### CAP-1 — Trustworthy column stats
Compute NaN-safe per-metric `means` and `stds` (shape `(3,)` each, no NaN), and
`missing_per_metric` (shape `(3,)`).


In [ ]:
means = None               # TODO: np.nanmean over axis 0
stds = None                # TODO: np.nanstd over axis 0
missing_per_metric = None  # TODO: NaN count per metric

In [ ]:
check("CAP-1: means shape (3,), no NaN",
      lambda: means.shape == (3,) and not np.isnan(means).any())
check("CAP-1: stds shape (3,), no NaN",
      lambda: stds.shape == (3,) and not np.isnan(stds).any())
check("CAP-1: missing counts match the matrix",
      lambda: int(missing_per_metric.sum()) == int(np.isnan(eval_metrics).sum()))

### CAP-2 — Impute, then z-score normalize
Replace missing cells with that metric's mean (mean-imputation via `np.where`),
giving `filled`. Then z-score normalize with **keepdims** broadcasting:
`normed = (filled - means_kd) / stds_kd`. Verify each column of `normed` has mean
~0 and std ~1.


💡 **Hint.** `np.where(np.isnan(eval_metrics), means, eval_metrics)` broadcasts the
`(3,)` means across rows to fill only the NaN cells. Build `means_kd`/`stds_kd`
with `keepdims=True` so they're `(1, 3)`.


In [ ]:
filled = None      # TODO: np.where(isnan, means, eval_metrics)
means_kd = None    # TODO: column means of `filled`, keepdims -> (1, 3)
stds_kd = None     # TODO: column stds of `filled`, keepdims -> (1, 3)
normed = None      # TODO: (filled - means_kd) / stds_kd

In [ ]:
check("CAP-2: filled has no NaN", lambda: not np.isnan(filled).any())
check("CAP-2: normed columns mean ~0",
      lambda: bool(np.allclose(normed.mean(axis=0), 0.0, atol=1e-9)))
check("CAP-2: normed columns std ~1",
      lambda: bool(np.allclose(normed.std(axis=0), 1.0, atol=1e-6)))

### CAP-3 — Composite score, flagging & ranking
Build a per-response `composite` = mean across the 3 **filled** metrics. Flag the
weak responses: those whose composite is **below the median**. Then rank everyone
descending and take the `top10` response indices.


In [ ]:
composite = None    # TODO: filled.mean(axis=1)  -> (200,)
threshold = None    # TODO: the median composite
weak_mask = None     # TODO: composite < threshold
weak_idx = None      # TODO: positions of weak responses (np.where)
top10 = None         # TODO: indices of the 10 highest composites, descending

In [ ]:
check("CAP-3: composite shape (200,)", lambda: composite.shape == (200,))
check("CAP-3: ~half are below the median",
      lambda: abs(weak_idx.size - 100) <= 1)
check("CAP-3: top10 sorted descending",
      lambda: bool(np.all(np.diff(composite[top10]) <= 0)))
check("CAP-3: best response is top10[0]",
      lambda: int(top10[0]) == int(np.argmax(composite)))

### CAP-4 — Cosine-similarity retrieval  *(the AI payoff)*
Embeddings let you ask "which responses are most **similar**?" Cosine similarity
is the dot product of **L2-normalized** vectors. Normalize every embedding to unit
length, then the full similarity matrix is just `unit @ unit.T`.

Use the best response (`query = top10[0]`) as the query and retrieve its 5 nearest
neighbors (excluding itself).


💡 **Hint.** `norms = np.linalg.norm(embeddings, axis=1, keepdims=True)` →
`unit = embeddings / norms`. The query's similarity row is `sim[query]`; set
`sim[query, query] = -np.inf` before taking the top 5 so it can't retrieve itself.


In [ ]:
norms = None         # TODO: L2 norm per row, keepdims -> (200, 1)
unit = None          # TODO: embeddings / norms  (unit vectors)
sim = None           # TODO: unit @ unit.T  -> (200, 200) cosine sims
query = None         # TODO: top10[0]

# Retrieve 5 nearest neighbors of `query`, excluding itself:
neighbors = None     # TODO: indices of the 5 highest sim[query], excluding query

In [ ]:
check("CAP-4: every embedding normalized to unit length",
      lambda: bool(np.allclose(np.linalg.norm(unit, axis=1), 1.0)))
check("CAP-4: sim diagonal is ~1 (self-similarity)",
      lambda: bool(np.allclose(np.diag(sim), 1.0)))
check("CAP-4: 5 neighbors, none is the query itself",
      lambda: neighbors.size == 5 and query not in neighbors.tolist())
check("CAP-4: neighbors sorted by descending similarity",
      lambda: bool(np.all(np.diff(sim[query, neighbors]) <= 0)))

### CAP-5 — Persist the results bundle
Save the analysis to a compressed `.npz` so it round-trips. Bundle `composite`,
`top10`, `weak_idx`, and `sim`, then reload and confirm `top10` survived intact.


In [ ]:
out_path = OUT / "eval_results.npz"
# TODO: np.savez_compressed(out_path, composite=..., top10=..., weak_idx=..., sim=...)

# Reload and verify:
loaded = None        # TODO: np.load(out_path)
top10_roundtrip = None  # TODO: loaded["top10"]

In [ ]:
check("CAP-5: bundle has all four arrays",
      lambda: set(loaded.keys()) == {"composite", "top10", "weak_idx", "sim"})
check("CAP-5: top10 round-trips exactly",
      lambda: np.array_equal(top10_roundtrip, top10))

## Wrap-up — what you can now do

- Create seeded `Generator`s with `np.random.default_rng` and explain why the global `np.random.*` API is discouraged.
- Draw from uniform, Gaussian, and integer distributions, and sample/shuffle reproducibly.
- Produce independent, reproducible parallel streams with `spawn`.
- Run a full eval pipeline end-to-end: NaN-safe stats → impute → z-score normalize → composite → flag → rank → cosine-similarity retrieve → save.

**That's Day 2.** You can now reason about an ndarray's shape, dtype, and memory;
vectorize and broadcast instead of looping; index with views/masks/fancy
selection; reduce, sort, and rank with NaN-safe stats; and generate reproducible
randomness — the NumPy foundation the rest of the Academy builds on.
